# Last.fm batch recommendations

Run `recommend_album()` over a subset of `albums.csv` and write results to `datasets/lastfm_recommendations_<subset>_<strategy>.csv`.

Re-run the batch cell to resume — rows with `status` already set are skipped.

In [11]:
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

from lastfm_albums import recommend_album, set_request_delay

In [12]:
DATA_DIR = Path("../datasets")
SOURCE_PATH = DATA_DIR / "albums.csv"

TRACK_SELECTION = "top_listener"  # top_listener | random | top_n_tracks
TOP_N = 3
N_RECS = 1
FETCH_FLOOR = 15
REQUEST_DELAY_SEC = 1
RANDOM_SEED = 42
SAVE_EVERY = 10

## Subset

Pick which albums to process. Output file name follows the subset key and `TRACK_SELECTION`.

In [13]:
SUBSETS = {
    "all": lambda df: df,
    "3plus": lambda df: df.loc[df["review_count"] > 2],
    "random_1k": lambda df: df.sample(1000),
}

In [14]:
SUBSET = "random_1k" 
MAX_ALBUMS = None  # e.g. 10 for a dry run

In [18]:
albums = SUBSETS[SUBSET](pd.read_csv(SOURCE_PATH))
if MAX_ALBUMS is not None:
    albums = albums.head(MAX_ALBUMS)

strategy_key = (
    f"top_n_tracks_{TOP_N}" if TRACK_SELECTION == "top_n_tracks" else TRACK_SELECTION
)
OUTPUT_PATH = DATA_DIR / f"lastfm_recommendations_{SUBSET}_{strategy_key}.csv"
print(f"Subset: {SUBSET} ({len(albums):,} albums)")
print(f"Strategy: {TRACK_SELECTION}")
print(f"Output: {OUTPUT_PATH}")

Subset: random_1k (1,000 albums)
Output: ../datasets/lastfm_recommendations_random_1k.csv


In [19]:
RESULT_COLS = ["strategy", "status", "error", "seed_track"] + [
    f"rec_{rank}_{field}"
    for rank in range(1, N_RECS + 1)
    for field in ("album", "artist", "score")
]

if OUTPUT_PATH.exists():
    work = pd.read_csv(OUTPUT_PATH)
else:
    work = albums.assign(**{col: pd.NA for col in RESULT_COLS})

for col in RESULT_COLS:
    if col not in work.columns:
        work[col] = pd.NA

work["strategy"] = work["strategy"].astype("string")
work.loc[work["strategy"].isna() | (work["strategy"].str.strip() == ""), "strategy"] = TRACK_SELECTION

work["status"] = work["status"].astype("string")
work["error"] = work["error"].astype("string")
work["seed_track"] = work["seed_track"].astype("string")
for rank in range(1, N_RECS + 1):
    work[f"rec_{rank}_album"] = work[f"rec_{rank}_album"].astype("string")
    work[f"rec_{rank}_artist"] = work[f"rec_{rank}_artist"].astype("string")
    work[f"rec_{rank}_score"] = pd.to_numeric(work[f"rec_{rank}_score"], errors="coerce")

pending = work.index[work["status"].isna() | (work["status"].astype(str).str.strip() == "")]
print(f"Strategy: {TRACK_SELECTION}")
print(f"Pending: {len(pending):,} / {len(work):,}")
work.head(3)

Pending: 1,000 / 1,000


,album_id,artist,album,review_count,status,error,seed_track,rec_1_album,rec_1_artist,rec_1_score
7375,7454,ElekTro4,Keystroke One,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
24827,24993,The Fratellis,Here We Stand,2,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
10577,10684,Hurt Valley,Glacial Pace,1,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [20]:
set_request_delay(REQUEST_DELAY_SEC)

since_save = 0
for idx in tqdm(pending, desc="Last.fm batch"):
    row = work.loc[idx]
    album_seed = RANDOM_SEED + int(row["album_id"])
    work.loc[idx, "strategy"] = TRACK_SELECTION

    try:
        _, seed_track, recs, _ = recommend_album(
            row["album"],
            artist=row["artist"],
            n_recs=N_RECS,
            fetch_floor=FETCH_FLOOR,
            track_selection=TRACK_SELECTION,
            top_n=TOP_N,
            random_seed=album_seed,
            clear_cache=False,
        )
        if recs.empty:
            work.loc[idx, ["status", "error"]] = ["no_results", pd.NA]
        else:
            work.loc[idx, "status"] = "ok"
            work.loc[idx, "error"] = pd.NA
            work.loc[idx, "seed_track"] = seed_track["name"] if seed_track else pd.NA
            for rank, (_, rec) in enumerate(recs.iterrows(), start=1):
                if rank > N_RECS:
                    break
                work.loc[idx, f"rec_{rank}_album"] = rec["album"]
                work.loc[idx, f"rec_{rank}_artist"] = rec["artist"]
                work.loc[idx, f"rec_{rank}_score"] = rec["similarity_score"]
    except Exception as exc:
        work.loc[idx, ["status", "error"]] = ["error", str(exc)[:500]]

    since_save += 1
    if since_save >= SAVE_EVERY:
        work.to_csv(OUTPUT_PATH, index=False)
        since_save = 0

work.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

Last.fm batch:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
print(work["status"].value_counts(dropna=False))
work.loc[work["status"] == "ok", ["artist", "album", "seed_track", "rec_1_album", "rec_1_artist"]].head()

status
ok            5
error         3
no_results    2
Name: count, dtype: int64


,artist,album,seed_track,rec_1_album,rec_1_artist
11421,Japanther,Leather Wings,[Untitled],Party Jail,Ed Schrader's Music Beat
57,1990s,Cookies,Cult Status,Comfort In Sound,Feeder
24091,The Beatles,Yellow Submarine,All Together Now,All the Best!,Paul McCartney & Wings
19150,Pink Floyd,The Early Years 1965-1972,Point Me at the Sky,Barrett,Syd Barrett
6260,Deerhoof,Holdypaws,The Great Car Tomb,The Air Force,Xiu Xiu
